In [2]:
import pandas as pd
import unicodedata
import re
from difflib import SequenceMatcher
from itertools import combinations
from pathlib import Path

In [3]:
#Visualización de las primeras filas 

DATA_PATH = Path('/Users/valemoravale/Documents/UNIVERSIDAD /Semestre 5/ETL/workshop3/data/2016.csv')
df = pd.read_csv(DATA_PATH)
df.head()

,Country,Region,Happiness Rank,Happiness Score,Lower Confidence Interval,Upper Confidence Interval,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption),Generosity,Dystopia Residual
0,Denmark,Western Europe,1,7.526,7.460,7.592,1.44178,1.16374,0.79504,0.57941,0.44453,0.36171,2.73939
1,Switzerland,Western Europe,2,7.509,7.428,7.590,1.52733,1.14524,0.86303,0.58557,0.41203,0.28083,2.69463
2,Iceland,Western Europe,3,7.501,7.333,7.669,1.42666,1.18326,0.86733,0.56624,0.14975,0.47678,2.83137
3,Norway,Western Europe,4,7.498,7.421,7.575,1.57744,1.12690,0.79579,0.59609,0.35776,0.37895,2.66465
4,Finland,Western Europe,5,7.413,7.351,7.475,1.40598,1.13464,0.81091,0.57104,0.41004,0.25492,2.82596


In [4]:
#Numero de filas, columnas y tipo de cada una
print('Rows:', len(df))
print('Columns:', df.shape[1])
df.info()

Rows: 157
Columns: 13
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Country                        157 non-null    object 
 1   Region                         157 non-null    object 
 2   Happiness Rank                 157 non-null    int64  
 3   Happiness Score                157 non-null    float64
 4   Lower Confidence Interval      157 non-null    float64
 5   Upper Confidence Interval      157 non-null    float64
 6   Economy (GDP per Capita)       157 non-null    float64
 7   Family                         157 non-null    float64
 8   Health (Life Expectancy)       157 non-null    float64
 9   Freedom                        157 non-null    float64
 10  Trust (Government Corruption)  157 non-null    float64
 11  Generosity                     157 non-null    float64
 12  Dystopia Residual           

In [5]:
#Valores faltantes y su porcentaje 
missing = df.isna().sum().to_frame('missing_count')
missing['missing_percent'] = (missing['missing_count'] / len(df) * 100).round(2)
missing

,missing_count,missing_percent
Country,0,0.0
Region,0,0.0
Happiness Rank,0,0.0
Happiness Score,0,0.0
Lower Confidence Interval,0,0.0
Upper Confidence Interval,0,0.0
Economy (GDP per Capita),0,0.0
Family,0,0.0
Health (Life Expectancy),0,0.0
Freedom,0,0.0


In [6]:
#Columnas duplicadas y Country duplicado 
print('Full duplicated rows:', df.duplicated().sum())
print('Duplicated Country:', df['Country'].duplicated().sum())

Full duplicated rows: 0
Duplicated Country: 0


In [7]:
# Crear función para limpiar nombres
def limpiar_texto(texto):
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"[^a-z0-9\s]", "", texto)
    texto = re.sub(r"\s+", " ", texto)
    return texto

# Crear columna limpia
df["country_clean"] = df["Country"].apply(limpiar_texto)

# Buscar duplicados exactos después de limpiar
duplicados_limpios = df[df.duplicated("country_clean", keep=False)]

print("Duplicados después de limpiar:")
print(duplicados_limpios[["Country", "country_clean"]])

Duplicados después de limpiar:
Empty DataFrame
Columns: [Country, country_clean]
Index: []


In [8]:
countries = df["Country"].dropna().unique()

posibles_duplicados = []

for pais1, pais2 in combinations(countries, 2):
    pais1_clean = limpiar_texto(pais1)
    pais2_clean = limpiar_texto(pais2)

    similitud = SequenceMatcher(None, pais1_clean, pais2_clean).ratio()

    if similitud >= 0.85:
        posibles_duplicados.append({
            "Country 1": pais1,
            "Country 2": pais2,
            "Similitud": round(similitud, 3)
        })

posibles_duplicados_df = pd.DataFrame(posibles_duplicados)

print(posibles_duplicados_df)

   Country 1 Country 2  Similitud
0    Iceland   Ireland      0.857
1  Australia   Austria      0.875


In [9]:
def limpiar_texto(texto):
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"[^a-z0-9\s]", "", texto)
    texto = re.sub(r"\s+", " ", texto)
    return texto

# Crear columna limpia para Region
df["region_clean"] = df["Region"].apply(limpiar_texto)

# Ver regiones originales y limpias
print(df[["Region", "region_clean"]].drop_duplicates())

                             Region                     region_clean
0                    Western Europe                   western europe
5                     North America                    north america
7         Australia and New Zealand        australia and new zealand
10  Middle East and Northern Africa  middle east and northern africa
13      Latin America and Caribbean      latin america and caribbean
21                Southeastern Asia                southeastern asia
26       Central and Eastern Europe       central and eastern europe
34                     Eastern Asia                     eastern asia
65               Sub-Saharan Africa                subsaharan africa
83                    Southern Asia                    southern asia


In [10]:
regions = df["Region"].dropna().unique()

posibles_duplicados_region = []

for region1, region2 in combinations(regions, 2):
    region1_clean = limpiar_texto(region1)
    region2_clean = limpiar_texto(region2)

    similitud = SequenceMatcher(None, region1_clean, region2_clean).ratio()

    if similitud >= 0.85:
        posibles_duplicados_region.append({
            "Region 1": region1,
            "Region 2": region2,
            "Similitud": round(similitud, 3)
        })

posibles_duplicados_region_df = pd.DataFrame(posibles_duplicados_region)

print(posibles_duplicados_region_df)

            Region 1       Region 2  Similitud
0  Southeastern Asia  Southern Asia      0.867


In [11]:
# Seleccionar solo columnas numéricas
numeric_cols = df.select_dtypes(include=["number"]).columns

ceros = (df[numeric_cols] == 0).sum()
negativos = (df[numeric_cols] < 0).sum()

print("Ceros por columna:")
print(ceros)
print("-------------------------------------")
print("Negativos por columna:")
print(negativos)

Ceros por columna:
Happiness Rank                   0
Happiness Score                  0
Lower Confidence Interval        0
Upper Confidence Interval        0
Economy (GDP per Capita)         1
Family                           1
Health (Life Expectancy)         1
Freedom                          1
Trust (Government Corruption)    1
Generosity                       1
Dystopia Residual                0
dtype: int64
-------------------------------------
Negativos por columna:
Happiness Rank                   0
Happiness Score                  0
Lower Confidence Interval        0
Upper Confidence Interval        0
Economy (GDP per Capita)         0
Family                           0
Health (Life Expectancy)         0
Freedom                          0
Trust (Government Corruption)    0
Generosity                       0
Dystopia Residual                0
dtype: int64


In [14]:
filas_no_validas = df[df[numeric_cols].le(0).any(axis=1)]

print(filas_no_validas)

                    Country                      Region  Happiness Rank  \
75                  Somalia          Sub-Saharan Africa              76   
86   Bosnia and Herzegovina  Central and Eastern Europe              87   
98                   Greece              Western Europe              99   
110            Sierra Leone          Sub-Saharan Africa             111   
132                   Sudan          Sub-Saharan Africa             133   
154                    Togo          Sub-Saharan Africa             155   

     Happiness Score  Lower Confidence Interval  Upper Confidence Interval  \
75             5.440                      5.321                      5.559   
86             5.163                      5.063                      5.263   
98             5.033                      4.935                      5.131   
110            4.635                      4.505                      4.765   
132            4.139                      3.928                      4.350   
154   

In [15]:
#ydata-profiling
from ydata_profiling import ProfileReport
profile = ProfileReport (df, title="Raw sale data profiling report", explorative= True)
profile.to_file("raw_sale_data_data_profile2016.html" )

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Export report to file: 100%|██████████| 1/1 [00:00<00:00, 249.30it/s]
